# Gini index - pull, clean, export

Indicator: Gini index of income inequality (0 = perfect equality, 1 = perfect inequality).
Census ACS5 Table B19083, variable B19083_001E.

**Different shape from the other notebooks on purpose:** Gini moves slowly enough that a
multi-year line chart mostly shows margin-of-error noise, not real change. This pulls one year
only and exports a flat value-per-geography schema for a bar chart, not a time series.

Same `CENSUS_API_KEY` as the other Census pulls.

In [ ]:
import json
import os
import time
from datetime import date
from pathlib import Path

import requests

API_KEY = os.environ.get("CENSUS_API_KEY")
if not API_KEY:
    raise RuntimeError("CENSUS_API_KEY not set. Set it as an environment variable, do not hardcode it here.")

RAW_DIR = Path("../raw")
DATA_DIR = Path("../data")
RAW_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

today = date.today().isoformat()
print("Pipeline run date:", today)

In [ ]:
VARIABLE = "B19083_001E"
DATASET = "acs/acs5"
YEAR = 2022  # most recent ACS5 vintage at time of writing; bump this each time you re-run

GEOGRAPHIES = {
    "Allegheny County": {"for": "county:003", "in": "state:42"},
    "Pennsylvania": {"for": "state:42"},
    "United States": {"for": "us:1"},
}

In [ ]:
values = {}
raw_responses = {}
url = f"https://api.census.gov/data/{YEAR}/{DATASET}"

for geo_name, geo_params in GEOGRAPHIES.items():
    params = {"get": f"NAME,{VARIABLE}", "key": API_KEY, **geo_params}
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
    except Exception as exc:
        print(f"FAILED  geo={geo_name}  error={exc}")
        continue

    raw_responses[geo_name] = payload
    header, data_row = payload[0], payload[1]
    record = dict(zip(header, data_row))
    value_raw = record.get(VARIABLE)

    value = None
    if value_raw is not None:
        try:
            value_f = float(value_raw)
            value = None if value_f < 0 else value_f
        except ValueError:
            value = None

    values[geo_name] = value
    time.sleep(0.2)

with open(RAW_DIR / f"gini_index_raw_{today}.json", "w") as f:
    json.dump(raw_responses, f, indent=2)

values

In [ ]:
missing = [g for g, v in values.items() if v is None]
if missing:
    print(f"Missing values for: {missing} - check before exporting")

In [ ]:
payload = {
    "indicator": "Gini index of income inequality",
    "geographies": list(GEOGRAPHIES.keys()),
    "unit": "index (0 to 1)",
    "year": YEAR,
    "snapshot_as_of": today,
    "status": "live",
    "source": f"U.S. Census Bureau, American Community Survey 5-Year Estimates, Table B19083, variable {VARIABLE}",
    "values": values,
    "notes": "Single most recent year only - Gini moves too slowly for a meaningful multi-year line chart at this margin of error.",
}

with open(DATA_DIR / "gini_index.json", "w") as f:
    json.dump(payload, f, indent=2)

print("Wrote", DATA_DIR / "gini_index.json")
payload